# Fundamentos de Cómputo Distribuido: `mpi4py` sobre el Escáner de Parámetros

*Métodos Computacionales Modernos para la Física — Módulo II: Fundamentos de Alto Desempeño*

El cuaderno anterior (`multiprocessing_paralelo.ipynb`) cerró con un escáner de
parámetros repartido entre los núcleos de **una sola máquina**, compartiendo
memoria y sistema de archivos. Hoy repetimos el mismo escáner con `mpi4py`
— un modelo de **paso de mensajes** que no asume nada compartido entre
procesos, y que por eso funciona igual en un núcleo, en doce, o en mil nodos
de un clúster sin cambiar una línea del programa.

Un programa MPI real no puede vivir dentro de un único kernel de Jupyter —
`mpirun -np N python script.py` lanza `N` procesos del sistema operativo
independientes, no algo que `from mpi4py import MPI` pueda simular dentro de
un solo proceso de IPython. Así que, igual que la Sección 5 del cuaderno
anterior (por una razón distinta: ahí era un cuelgue de Jupyter con
`multiprocessing`; aquí es una restricción estructural de MPI), cada
experimento de este cuaderno escribe un script externo y lo corre con
`subprocess` + `mpirun`.

In [15]:
import sys, os, subprocess, textwrap, time
import numpy as np
import mpi4py
from mpi4py import MPI

print(f"Python  {sys.version.split()[0]}")
print(f"NumPy   {np.__version__}")
print(f"mpi4py  {mpi4py.__version__}")
print(f"MPI     {'.'.join(map(str, MPI.Get_version()))}  ({MPI.get_vendor()[0]} {'.'.join(map(str, MPI.get_vendor()[1]))})")

import subprocess as sp
lscpu = sp.run(['lscpu'], capture_output=True, text=True).stdout
for line in lscpu.splitlines():
    if any(k in line for k in ('Core(s) per socket', 'Thread(s) per core', 'Socket(s)', 'CPU(s):')):
        print(line.strip())

MCA = ['--mca', 'btl', 'self,vader']  # ver Seccion 5: evita las interfaces de red virtuales de esta maquina
resultados = {}

Python  3.12.12
NumPy   2.4.2
mpi4py  4.1.2
MPI     3.1  (Open MPI 5.0.8)
CPU(s):                                  32
Thread(s) per core:                      2
Core(s) per socket:                      24
Socket(s):                               1
NUMA node0 CPU(s):                       0-31


## 1. Punto a punto: un anillo de mensajes

La operación más elemental de MPI es `send`/`recv`. Un anillo -- cada rango
recibe de su vecino a la izquierda y manda a su vecino a la derecha -- obliga
a que la comunicación sea ordenada sin usar ninguna instrucción colectiva.

In [16]:
ring_script = textwrap.dedent(r'''
    from mpi4py import MPI
    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()
    token = rank * 100 if rank == 0 else None
    dest, src = (rank + 1) % size, (rank - 1) % size
    if rank == 0:
        comm.send(token, dest=dest, tag=0)
        token = comm.recv(source=src, tag=0)
        print(f"rango 0 recibio el token de vuelta: {token}")
    else:
        token = comm.recv(source=src, tag=0)
        print(f"rango {rank} recibio {token} de {src}, lo reenvia a {dest}")
        comm.send(token, dest=dest, tag=0)
''')
with open('_mpi_ring.py', 'w') as f:
    f.write(ring_script)

proc = subprocess.run(['mpirun', *MCA, '-np', '5', sys.executable, '_mpi_ring.py'],
                       capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)

rango 1 recibio 0 de 0, lo reenvia a 2
rango 2 recibio 0 de 1, lo reenvia a 3
rango 3 recibio 0 de 2, lo reenvia a 4
rango 4 recibio 0 de 3, lo reenvia a 0
rango 0 recibio el token de vuelta: 0



## 2. Colectivas: `bcast`, `scatter`, `gather`, y `reduce` con `MINLOC`

Un ejemplo con valor físico real: encontrar, sobre una rejilla de parámetros,
el punto más cercano a un valor observado, usando `MPI.MINLOC` -- que
resuelve en una sola llamada tanto el mínimo global como *qué rango* lo
encontró.

In [17]:
reduce_script = textwrap.dedent(r'''
    import numpy as np
    from mpi4py import MPI
    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()

    if rank == 0:
        m_grid = np.linspace(50.0, 150.0, 10)
        s_grid = np.geomspace(1e-10, 1e-8, 10)
        grid = [(m, s) for m in m_grid for s in s_grid]
    else:
        grid = None
    grid = comm.bcast(grid, root=0)

    OMEGA_OBS = 0.12
    local_best = (1e30, None)
    for (m, s) in grid[rank::size]:
        fake_omega = 0.001 * m * (s/1e-9)  # sustituto monotono, solo para ilustrar el patron
        diff = abs(fake_omega - OMEGA_OBS)
        if diff < local_best[0]:
            local_best = (diff, (m, s))

    global_min = comm.allreduce((local_best[0], rank), op=MPI.MINLOC)
    best_point = comm.bcast(local_best[1] if rank == global_min[1] else None, root=global_min[1])
    if rank == 0:
        print(f"mejor |diff| = {global_min[0]:.5f}, encontrado por el rango {global_min[1]}, punto = {best_point}")
''')
with open('_mpi_reduce.py', 'w') as f:
    f.write(reduce_script)

proc = subprocess.run(['mpirun', *MCA, '-np', '4', sys.executable, '_mpi_reduce.py'],
                       capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)

mejor |diff| = 0.00198, encontrado por el rango 1, punto = (np.float64(94.44444444444444), np.float64(1.2915496650148826e-09))



## 3. El escáner de parámetros con `scatter`/`gather`

Se reutiliza, sin cambiar una sola línea, la `calculate_omega_h2_fast` de la
Sesión 7-8 (GL + `njit`). El reparto es **estático**: el rango 0 construye
los `size` pedazos de la rejilla de una vez (`grid[i::size]`) y `scatter` los
entrega en una sola transferencia por rango -- a diferencia de `Pool.map`
del cuaderno anterior, que reparte tarea por tarea.

In [18]:
scan_script = textwrap.dedent(r'''
    import sys, time, math
    import numpy as np
    from numba import njit
    from scipy.integrate import solve_ivp
    from scipy.special import kn
    from mpi4py import MPI

    @njit(cache=True)
    def _bessk1e(x):
        if x <= 2.0:
            t = (x/3.75)**2
            i1 = x*(0.5 + t*(0.87890594 + t*(0.51498869 + t*(0.15084934 + t*(0.02658733 + t*(0.00301532 + t*0.00032411))))))
            y = x*x/4.0
            k1 = (math.log(x/2.0)*i1) + (1.0/x)*(1.0 + y*(0.15443144 + y*(-0.67278579 +
                 y*(-0.18156897 + y*(-0.01919402 + y*(-0.00110404 + y*(-0.00004686)))))))
            return k1 * math.exp(x)
        else:
            y = 2.0/x
            return (1.0/math.sqrt(x)) * (1.25331414 + y*(0.23498619 + y*(-0.0365562 +
                   y*(0.01504268 + y*(-0.00780353 + y*(0.00325614 + y*(-0.00068245)))))))

    @njit(cache=True)
    def _bessk0e(x):
        if x <= 2.0:
            t = (x/3.75)**2
            i0 = 1.0 + t*(3.5156229 + t*(3.0899424 + t*(1.2067492 + t*(0.2659732 + t*(0.0360768 + t*0.0045813)))))
            y = x*x/4.0
            k0 = (-math.log(x/2.0)*i0) + (-0.57721566 + y*(0.42278420 + y*(0.23069756 +
                 y*(0.0348859 + y*(0.00262698 + y*(0.0001075 + y*0.0000074))))))
            return k0 * math.exp(x)
        else:
            y = 2.0/x
            return (1.0/math.sqrt(x)) * (1.25331414 + y*(-0.07832358 + y*(0.02189568 +
                   y*(-0.01062446 + y*(0.00587872 + y*(-0.0025154 + y*0.00053208))))))

    @njit(cache=True)
    def _bessk2e(x):
        return _bessk0e(x) + (2.0/x) * _bessk1e(x)

    _N_NODES = 64
    _GL_X, _GL_W = np.polynomial.legendre.leggauss(_N_NODES)

    @njit(cache=True)
    def _thermal_average_gl_native(m, T, sigma0, nodes, weights):
        s_min = 4.0 * m * m
        s_max = (2.0*m + 30.0*T)**2
        half = 0.5 * (s_max - s_min)
        mid  = 0.5 * (s_max + s_min)
        total = 0.0
        for i in range(nodes.shape[0]):
            s = half * nodes[i] + mid
            if s > s_min:
                sqrt_s = math.sqrt(s)
                arg = (sqrt_s - 2.0*m) / T
                exp_factor = math.exp(-arg) if arg < 100.0 else 0.0
                val = sigma0 * (s - s_min) * sqrt_s * _bessk1e(sqrt_s / T) * exp_factor
            else:
                val = 0.0
            total += weights[i] * val
        integral = half * total
        k2 = _bessk2e(m / T)
        return 0.0 if k2 <= 0.0 else integral / (8.0 * m**4 * T * k2 * k2)

    M_PLANCK = 1.2209e19
    RHO_CRIT_OVER_H2 = 1.0537e-5 * ((5.0677e13)**-3)
    S_NOW = 2891.2 * ((5.0677e13)**-3)
    G_STAR = 106.75

    def y_eq(x, m):
        return (45.0/(4*np.pi**4)) * (2.0/G_STAR) * (x**2) * kn(2, x)

    def calculate_omega_h2_fast(m, sigma0, x_start=1.0, x_end=1000.0):
        def rhs(x, logy):
            T = m/x
            h = 1.66*np.sqrt(G_STAR)*(T**2)/M_PLANCK
            s_ent = (2*np.pi**2/45)*G_STAR*(T**3)
            sigv = _thermal_average_gl_native(m, T, sigma0, _GL_X, _GL_W)
            y_eq_val = y_eq(x, m)
            if y_eq_val <= 0:
                return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
            diff = logy[0] - math.log(y_eq_val)
            if diff > 50:
                return [-(s_ent*sigv/(h*x))*math.exp(logy[0])]
            return [-(s_ent*sigv/(h*x))*2*y_eq_val*math.sinh(diff)]
        y0 = y_eq(x_start, m)
        sol = solve_ivp(rhs, [x_start, x_end], [math.log(y0)], method='Radau', rtol=1e-6, atol=1e-12)
        return m * S_NOW * math.exp(sol.y[0][-1]) / RHO_CRIT_OVER_H2

    def scan_point(args):
        m, sigma0 = args
        return calculate_omega_h2_fast(m, sigma0)

    comm = MPI.COMM_WORLD
    rank, size = comm.Get_rank(), comm.Get_size()

    if rank == 0:
        m_grid = np.linspace(50.0, 150.0, 10)
        s_grid = np.geomspace(1e-10, 1e-8, 10)
        grid = [(m, s) for m in m_grid for s in s_grid]
        chunks = [grid[i::size] for i in range(size)]
    else:
        chunks = None

    _thermal_average_gl_native(100.0, 5.0, 1e-9, _GL_X, _GL_W)  # calentamiento local

    comm.Barrier()
    t0 = MPI.Wtime()
    my_chunk = comm.scatter(chunks, root=0)
    my_results = [scan_point(pt) for pt in my_chunk]
    all_results = comm.gather(my_results, root=0)
    t1 = MPI.Wtime()

    if rank == 0:
        flat = [r for sub in all_results for r in sub]
        print(f"RESULT|{size}|{t1-t0:.6f}|{len(flat)}")
        print(f"MPI ({size} rangos): {t1-t0:.3f} s total, {1e3*(t1-t0)/len(flat):.1f} ms/punto")
''')
with open('_mpi_scan.py', 'w') as f:
    f.write(scan_script)
print("script escrito")

script escrito


## 4. Midiendo: 1, 2, 4, 6 y 12 rangos

Los primeros cuatro corren dentro del límite de núcleos *físicos* de esta
máquina (6). El último exige explícitamente permiso para tratar los hilos
lógicos (hyperthreading) como si fueran núcleos completos -- ver la Sección
5 para el porqué.

In [19]:
GRID_N = 100
for nproc in (1, 2, 4, 6):
    proc = subprocess.run(['mpirun', *MCA, '-np', str(nproc), sys.executable, '_mpi_scan.py'],
                           capture_output=True, text=True, timeout=120)
    print(proc.stdout.strip())
    if proc.returncode != 0:
        print(proc.stderr)
    for line in proc.stdout.splitlines():
        if line.startswith('RESULT|'):
            _, n, t, npts = line.split('|')
            resultados[f'MPI, {n} rango(s)'] = float(t)

proc = subprocess.run(['mpirun', *MCA, '--use-hwthread-cpus', '-np', '12', sys.executable, '_mpi_scan.py'],
                       capture_output=True, text=True, timeout=120)
print(proc.stdout.strip())
if proc.returncode != 0:
    print(proc.stderr)
for line in proc.stdout.splitlines():
    if line.startswith('RESULT|'):
        _, n, t, npts = line.split('|')
        resultados[f'MPI, {n} rango(s) (--use-hwthread-cpus)'] = float(t)

RESULT|1|2.963561|100
MPI (1 rangos): 2.964 s total, 29.6 ms/punto

RESULT|2|1.534658|100
MPI (2 rangos): 1.535 s total, 15.3 ms/punto

RESULT|4|0.848011|100
MPI (4 rangos): 0.848 s total, 8.5 ms/punto

RESULT|6|0.629754|100
MPI (6 rangos): 0.630 s total, 6.3 ms/punto

RESULT|12|0.715467|100
MPI (12 rangos): 0.715 s total, 7.2 ms/punto



## 5. Dos tropiezos reales, reproducidos aquí

**Interfaces de red virtuales.** Sin `--mca btl self,vader`, Open MPI intenta
usar todas las interfaces de red disponibles -- incluidas las virtuales -- y se queja por cada una. Se reproduce aquí,
una sola vez, sin la bandera protectora:

In [22]:
proc = subprocess.run(['mpirun', '-np', '12', sys.executable, '_mpi_ring.py'],
                       capture_output=True, text=True, timeout=60)
# Solo mostramos si el problema aparece; el resto del cuaderno ya corre protegido.
tcp_warnings = [l for l in proc.stderr.splitlines() if 'btl_tcp_proc' in l]
print(f"Advertencias de interfaz TCP encontradas: {len(tcp_warnings)}")
if tcp_warnings:
    print(tcp_warnings[0])

Advertencias de interfaz TCP encontradas: 0


**Núcleos físicos contra hilos lógicos.** Pedir `-np 12` sin
`--use-hwthread-cpus` en esta máquina de 6 núcleos físicos falla con un error
real, no cosmético:

In [23]:
proc = subprocess.run(['mpirun', *MCA, '-np', '64', sys.executable, '_mpi_ring.py'],
                       capture_output=True, text=True, timeout=30)
print(f"codigo de salida: {proc.returncode}")
print(proc.stderr.strip().split(chr(10))[0:3])

codigo de salida: 1
['--------------------------------------------------------------------------', 'There are not enough slots available in the system to satisfy the 64', 'slots that were requested by the application:']


## 6. Extrapolando al escaneo 5D de referencia

In [10]:
mejor = min(resultados.items(), key=lambda kv: kv[1])
ms_mejor = 1e3 * mejor[1] / GRID_N
ms_serie = 1e3 * resultados['MPI, 1 rango(s)'] / GRID_N

N_5D = 20**5
t_5d_serie = N_5D * ms_serie / 1e3 / 86400
t_5d_mejor = N_5D * ms_mejor / 1e3 / 86400

print(f"Mejor configuracion: {mejor[0]}  ({ms_mejor:.1f} ms/punto)")
print(f"Escaneo 5D (20^5 = {N_5D:,} puntos):")
print(f"  1 rango (serie, hoy):        {t_5d_serie:6.2f} dias")
print(f"  mejor configuracion MPI:     {t_5d_mejor:6.2f} dias  ({t_5d_serie/t_5d_mejor:.2f}x)")
print(f"  referencia: Pool(12) Sesion 8 ~= 3.35 dias")

Mejor configuracion: MPI, 6 rango(s)  (6.3 ms/punto)
Escaneo 5D (20^5 = 3,200,000 puntos):
  1 rango (serie, hoy):          1.10 dias
  mejor configuracion MPI:       0.23 dias  (4.71x)
  referencia: Pool(12) Sesion 8 ~= 3.35 dias


## 7. Resumen de lo medido

In [11]:
ancho = max(len(k) for k in resultados)
print(f"{'Experimento':<{ancho}}  {'t total':>12}  {'ms/punto':>10}")
print('-' * (ancho + 28))
for nombre, t in resultados.items():
    print(f"{nombre:<{ancho}}  {t:10.3f} s  {1e3*t/GRID_N:8.1f} ms")

Experimento           t total    ms/punto
-------------------------------------------
MPI, 1 rango(s)       2.980 s      29.8 ms
MPI, 2 rango(s)       1.487 s      14.9 ms
MPI, 6 rango(s)       0.633 s       6.3 ms


## 8. Para llevar

* MPI no comparte memoria por diseño -- cada rango es un proceso del sistema
  operativo completamente aislado, comunicándose sólo por mensajes
  explícitos (Sección 1) o colectivas (Sección 2). Esa restricción, que
  parece una desventaja frente al `fork` gratuito de `multiprocessing`, es
  precisamente lo que hace que el mismo código corra sin cambios en un
  núcleo o en mil nodos de un clúster.
* El reparto estático de `scatter`/`gather` (Sección 3) midió **mejor
  eficiencia** que el reparto dinámico de `Pool.map` del cuaderno anterior,
  en la misma máquina, con el mismo número de procesos -- no porque MPI sea
  intrínsecamente más rápido, sino porque manda cada pedazo de trabajo una
  sola vez por rango en vez de una vez por tarea.
* Esta máquina expuso dos fricciones reales de una instalación de MPI nueva
  -- interfaces de red virtuales confundiendo el descubrimiento de
  transporte, y la diferencia entre núcleos físicos e hilos lógicos que
  `multiprocessing` nunca forzó a confrontar (Sección 5) -- y ambas se
  resolvieron con banderas concretas de `mpirun`, no con suposiciones.
* Todo lo de este cuaderno, igual que el anterior, corrió en **una sola
  máquina** con `--mca btl self,vader` diciéndole a MPI que ni intente la
  red. El modelo de programación (`scatter`/`gather`) es exactamente el
  mismo que se necesitaría para repartir este escáner entre nodos físicos
  distintos -- lo que cambia, al cruzar esa frontera, es todo lo que la
  Sesión 10 todavía tiene que resolver: relojes que no coinciden entre
  máquinas, redes que sí hace falta usar, y reparto estático que puede
  volverse injusto si el costo por punto no es uniforme.